# Week 1 - MNIST: baseline, MLP, dropout
Marvel Kevin Nathanael | 00000108042

Mengikuti TLM 1 dan Chapter 1 Packt. Semua model memakai data dan preprocessing yang sama.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
tf.keras.utils.set_random_seed(SEED)
(x_dev, y_dev), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_dev = x_dev.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0
x_train, x_val, y_train, y_val = train_test_split(
    x_dev, y_dev, test_size=0.1, random_state=SEED, stratify=y_dev)
print(x_train.shape, x_val.shape, x_test.shape)

In [ ]:
def make_model(kind):
    layers = [tf.keras.Input(shape=(784,))]
    if kind != "baseline":
        layers.append(tf.keras.layers.Dense(128, activation="relu"))
        if kind == "dropout":
            layers.append(tf.keras.layers.Dropout(0.3))
        layers.append(tf.keras.layers.Dense(128, activation="relu"))
        if kind == "dropout":
            layers.append(tf.keras.layers.Dropout(0.3))
    layers.append(tf.keras.layers.Dense(10, activation="softmax"))
    model = tf.keras.Sequential(layers, name=kind)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

models, rows = {}, []
for kind in ["baseline", "mlp", "dropout"]:
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(kind)
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=8, batch_size=128, verbose=1,
                        callbacks=[tf.keras.callbacks.EarlyStopping(
                            monitor="val_accuracy", mode="max", patience=2,
                            restore_best_weights=True)])
    models[kind] = model
    rows.append({"model": kind, "best_val_accuracy": max(history.history["val_accuracy"])})
results = pd.DataFrame(rows).sort_values("best_val_accuracy", ascending=False)
display(results)

In [ ]:
best_name = results.iloc[0]["model"]
test_loss, test_accuracy = models[best_name].evaluate(x_test, y_test, verbose=0)
pred = np.argmax(models[best_name].predict(x_test[:10], verbose=0), axis=1)
print(f"Model: {best_name} | Test accuracy: {test_accuracy:.4f} | Test loss: {test_loss:.4f}")
print("Prediksi:", pred.tolist())
print("Label:", y_test[:10].tolist())

**Catatan:** Model dipilih lewat validasi, lalu diuji sekali pada test. MNIST belum mewakili tulisan tangan dari kamera ponsel.